# Consolidated Outpatient Base 2025 / TEMPLATE per state
## Predictive model of healthcare demand in supplementary health on North Region

This notebook belongs to the **raw → interim** step of the project data
pipeline. It downloads, merges, and documents the raw TISS Outpatient data
from ANS (DET + CONS + REM) for **one state, year 2025, without any
treatment, cleaning, or breast cancer filtering**. It is self-contained: it
runs from start to finish locally (or on Google Colab, after cloning the
repo), without depending on any pre-downloaded or pre-uploaded files.

After all states are concatenated into
a single North region base, the treatment (numerical/categorical typing, checking and
removal of duplicates) and filtering by TUSS codes and SBOC guidelines are
applied **only once**, over the complete base instead of 7 times, one per
state, which could cause a divergence in the treatment/filtering logic.

### How to adapt this notebook for another state

Just change the variable
`STATE` in the configuration cell (Phase 0). No other cell needs to be
altered; the file names, data dictionary, and provenance documentation are
dynamically generated from this variable.

### What this notebook produces

A file `{STATE}_ambulatorial_full_2025.parquet`
with one row per item/procedure (DET level), enriched with the event
context columns (CONS) and remuneration (REM), plus a provenance table
documenting the origin of each column — written into
`data/interim/ans_ambulatorial_by_state/{STATE}/`, ready for the merge
notebook (`02_merge_north_states.ipynb`) to pick up.

## Phase 0 — Configuration

In [1]:
import gc
import os
from pathlib import Path
import zipfile
import urllib.request

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# ----------------------------------------------------------------------------
# CHANGE BASED ON THE STATE YOU WANT
# ----------------------------------------------------------------------------
STATE = "AM"
# ----------------------------------------------------------------------------

YEAR = "2025"
MONTHS = [f"{m:02d}" for m in range(1, 13)]

BASE_URL = "https://dadosabertos.ans.gov.br/FTP/PDA/TISS/AMBULATORIAL"

# Resolve the repository root from either the repository root or notebooks/.
WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "ans_ambulatorial" / STATE
INTERIM_DIR = DATA_DIR / "interim" / "ans_ambulatorial_by_state" / STATE
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILENAME = f"{STATE}_ambulatorial_full_{YEAR}.parquet"
PROVENANCE_FILENAME = f"{STATE}_proveniencia_colunas.csv"
OUTPUT_PATH = INTERIM_DIR / OUTPUT_FILENAME
PROVENANCE_PATH = INTERIM_DIR / PROVENANCE_FILENAME

print(f"State: {STATE} | Year: {YEAR} | Months: {len(MONTHS)}")
print(f"Output will be written to: {OUTPUT_PATH}")

State: AM | Year: 2025 | Months: 12
Output will be written to: data\interim\ans_ambulatorial_by_state\AM\AM_ambulatorial_full_2025.parquet


## Phase 1 — Raw data acquisition

The ANS Outpatient TISS publishes 3 files per state/year/month:

| Table | Content | Granularity |
|---|---|---|
| **DET** | Each procedure/assistance item charged | 1 row per item (multiple rows per event) |
| **CONS** | Event context: who is the beneficiary, where was attended, what type of care | 1 row per event |
| **REM** | How much was paid/remunerated for that event | 1 or more rows per event (rarely more than 1) |

The 3 share the same key, `ID_EVENTO_ATENCAO_SAUDE`, which is what
allows merging the 3 tables later.

In [2]:
def download_file(url, destination):
    if not os.path.exists(destination):
        urllib.request.urlretrieve(url, destination)
    return destination


def read_csv_from_zip(zip_path, sep=";"):
    with zipfile.ZipFile(zip_path) as z:
        csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(csv_name) as f:
            return pd.read_csv(f, sep=sep, encoding="utf-8-sig", dtype=str)


zip_files = {"DET": {}, "CONS": {}, "REM": {}}

for m in MONTHS:
    for file_type in zip_files:
        fname = f"{STATE}_{YEAR}{m}_AMB_{file_type}.zip"
        url = f"{BASE_URL}/{YEAR}/{STATE}/{fname}"
        local_zip = os.path.join(RAW_DIR, fname)
        download_file(url, local_zip)
        zip_files[file_type][m] = local_zip
    print(f"[{STATE}] month {m}: DET + CONS + REM downloaded")

print(f"\nDownload completed: {len(MONTHS) * 3} files ({len(MONTHS)} months x 3 tables).")

[AM] month 01: DET + CONS + REM downloaded
[AM] month 02: DET + CONS + REM downloaded
[AM] month 03: DET + CONS + REM downloaded
[AM] month 04: DET + CONS + REM downloaded
[AM] month 05: DET + CONS + REM downloaded
[AM] month 06: DET + CONS + REM downloaded
[AM] month 07: DET + CONS + REM downloaded
[AM] month 08: DET + CONS + REM downloaded
[AM] month 09: DET + CONS + REM downloaded
[AM] month 10: DET + CONS + REM downloaded
[AM] month 11: DET + CONS + REM downloaded
[AM] month 12: DET + CONS + REM downloaded

Download completed: 36 files (12 months x 3 tables).


## Phase 2 — Data dictionary and provenance contract

Before merging anything, we declare **where each column of the final base
comes from** and **what analytical type it will have, when treated**
(numerical or categorical). This serves two purposes:

1. Automatically generates the column provenance table required by the
   deliverable;
   it never becomes outdated in relation to the code, because it's the same
   structure that drives the processing.
2. Provides a ready reference for the deliverable "identification of
   numerical and categorical columns" in `documentation.md` — the type
   CONVERSION itself (string -> float/category) DOES NOT happen in this
   notebook, only in the complete North region base. Here, every
   column comes out as it arrived from the original CSV: string.

Each line of the dictionary below documents: final column, origin table,
intended analytical type (for the future treatment stage), description
(based on the TISS Standard Variable Dictionary from ANS) and the
transformation applied IN THIS merging stage, if any, which in the vast
majority of cases is "None", precisely because this notebook only merges,
it doesn't treat.

In [3]:
PROVENANCE_DICTIONARY = [
    # --- from DET (base; item/procedure level) ---
    {"column": "ID_EVENTO_ATENCAO_SAUDE", "source": "DET", "type": "key",
     "description": "Unique identifier of the outpatient event.",
     "transformation": "None -- it's the key used to merge DET, CONS, and REM."},
    {"column": "UF_PRESTADOR", "source": "DET", "type": "categorical",
     "description": "UF of the provider who performed the procedure.",
     "transformation": "Also exists in CONS; we validated that the values are identical "
                       "(see Phase 3) and discarded the copy from CONS to avoid duplicating the column."},
    {"column": "DT_REALIZACAO", "source": "DET", "type": "categorical/temporal",
     "description": "Year-month of event occurrence, yyyy-mm format.",
     "transformation": "Same case as UF_PRESTADOR: identical to CONS, CONS copy discarded."},
    {"column": "CD_PROCEDIMENTO", "source": "DET", "type": "categorical",
     "description": "Procedure/assistance item code (TUSS tables 19, 20, 22 or 63).",
     "transformation": "None. It will be the TUSS/SBOC filter key in the North region stage."},
    {"column": "CD_TABELA_REFERENCIA", "source": "DET", "type": "categorical",
     "description": "TUSS reference table code for the procedure.",
     "transformation": "None."},
    {"column": "QT_ITEM_EVENTO_INFORMADO", "source": "DET", "type": "numerical",
     "description": "Quantity of the item/procedure reported by the provider.",
     "transformation": "None at this stage -- remains as string, with decimal comma (e.g., '9,25'), "
                       "same as the original CSV. Conversion to float (without rounding to integer: "
                       "~0.001% of values are fractional, probably partial medication dosage) "
                       "is for the North region base treatment. Comes null from "
                       "DET itself when IND_PACOTE=1 -- items charged within a package do not "
                       "have individual quantity/value reported."},
    {"column": "VL_ITEM_EVENTO_INFORMADO", "source": "DET", "type": "numerical",
     "description": "Reported value of the item/procedure.",
     "transformation": "None at this stage -- remains as string with decimal comma (e.g., '32,51'). "
                       "Comes null from DET itself when IND_PACOTE=1, for the same reason as "
                       "QT_ITEM_EVENTO_INFORMADO."},
    {"column": "VL_ITEM_PAGO_FORNECEDOR", "source": "DET", "type": "numerical",
     "description": "Value actually paid by the operator directly to the provider.",
     "transformation": "None at this stage -- remains as string with decimal comma. Comes null from "
                       "DET itself when IND_PACOTE=1."},
    {"column": "UNIDADE_MEDIDA", "source": "DET", "type": "categorical",
     "description": "Unit of measurement for the item, according to TUSS table 60.",
     "transformation": "None."},
    {"column": "IND_PACOTE", "source": "DET", "type": "binary categorical",
     "description": "Indicates whether the item is part of a procedure package (0=no, 1=yes).",
     "transformation": "Maintained as string, in the original ANS standard."},
    {"column": "IND_TABELA_PROPRIA", "source": "DET", "type": "binary categorical",
     "description": "Indicates whether the code was created by the operator itself (outside TUSS).",
     "transformation": "Maintained as string."},

    # --- from CONS (event context) ---
    {"column": "ID_PLANO", "source": "CONS", "type": "categorical",
     "description": "Plan identifier (valid only for this ANS publication).",
     "transformation": "None."},
    {"column": "FAIXA_ETARIA", "source": "CONS", "type": "categorical (ordinal)",
     "description": "Beneficiary's age group, in pre-defined bins by ANS.",
     "transformation": "None."},
    {"column": "SEXO", "source": "CONS", "type": "categorical",
     "description": "Beneficiary's sex, according to TUSS table 43.",
     "transformation": "None."},
    {"column": "CD_MUNICIPIO_BENEFICIARIO", "source": "CONS", "type": "categorical",
     "description": "IBGE code of the beneficiary's municipality of residence.",
     "transformation": "None."},
    {"column": "PORTE", "source": "CONS", "type": "categorical (ordinal)",
     "description": "Operator size by number of active beneficiaries (small/medium/large).",
     "transformation": "None."},
    {"column": "CD_MODALIDADE", "source": "CONS", "type": "categorical",
     "description": "Operator modality code (cooperative, self-management, insurer, etc.).",
     "transformation": "None."},
    {"column": "NM_MODALIDADE", "source": "CONS", "type": "categorical",
     "description": "Name of the operator modality.",
     "transformation": "None."},
    {"column": "CD_MUNICIPIO_PRESTADOR", "source": "CONS", "type": "categorical",
     "description": "IBGE code of the executive provider's municipality.",
     "transformation": "None."},
    {"column": "CD_CARATER_ATENDIMENTO", "source": "CONS", "type": "categorical",
     "description": "Nature of care: elective or urgency/emergency.",
     "transformation": "None."},
    {"column": "TIPO_ATENDIMENTO", "source": "CONS", "type": "categorical",
     "description": "Type of care, according to TUSS table 50.",
     "transformation": "None."},
    {"column": "REGIME_ATENDIMENTO", "source": "CONS", "type": "categorical",
     "description": "Type of care, according to TUSS table 76.",
     "transformation": "None."},
    {"column": "CD_MOTIVO_SAIDA", "source": "CONS", "type": "categorical",
     "description": "Reason for discharge, according to TUSS table 39.",
     "transformation": "None."},
    {"column": "CBO", "source": "CONS", "type": "categorical",
     "description": "Occupation of the performing professional (Brazilian Classification of Occupations).",
     "transformation": "None."},
    {"column": "TIPO_CONSULTA", "source": "CONS", "type": "categorical",
     "description": "Type of consultation, according to TUSS table 52.",
     "transformation": "None."},
    {"column": "SAUDE_OCUPACIONAL", "source": "CONS", "type": "categorical",
     "description": "Occupational health indicator, according to TUSS table 77.",
     "transformation": "None."},
    {"column": "IND_ACIDENTE_DOENCA", "source": "CONS", "type": "binary categorical",
     "description": "Indicates whether the care results from an accident or related illness.",
     "transformation": "Maintained as string."},
    {"column": "LG_VALOR_PREESTABELECIDO", "source": "CONS", "type": "binary categorical",
     "description": "Indicates whether remuneration is for a pre-established value (independent of services performed).",
     "transformation": "Maintained as string."},

    # --- from REM (remuneration, aggregated by event) ---
    {"column": "VL_TOTAL_REMUNERACAO", "source": "REM (aggregated)", "type": "numerical",
     "description": "Total remuneration of the event.",
     "transformation": "THE ONLY type conversion in this notebook (everything else remains string): "
                       "converted from string with decimal comma to float and SUMMED by "
                       "ID_EVENTO_ATENCAO_SAUDE before the merge, as ~0.3% of events have more "
                       "than one line in REM (more than one remuneration model for the same event). "
                       "Without this sum, the merge would duplicate the corresponding DET row -- this is a "
                       "structural necessity of the join, not a content cleaning."},
    {"column": "CD_MODELO_REMUNERACAO", "source": "REM (aggregated)", "type": "categorical",
     "description": "Remuneration model(s) applied to the event.",
     "transformation": "When there is more than one model for the same event, codes are concatenated with '|'."},
    {"column": "N_LINHAS_REM", "source": "REM (derived)", "type": "numerical",
     "description": "Number of REM rows aggregated for this event.",
     "transformation": "New column, created at this stage. Flags events with more than one "
                       "remuneration model (N_LINHAS_REM > 1); does not exist in the raw data."},

    # --- derived from the ingestion process itself ---
    {"column": "MES", "source": "derived (ingestion loop)", "type": "categorical",
     "description": "Competence (YYYYMM) of the source file.",
     "transformation": "New column, assigned from the downloaded filename in each loop iteration."},
    {"column": "UF", "source": "derived (ingestion loop)", "type": "categorical",
     "description": "Acronym of the state processed in this notebook.",
     "transformation": "New column. It is the column used to concatenate the 7 states of the North "
                       "region in the next stage of the project."},
]

provenance_df = pd.DataFrame(PROVENANCE_DICTIONARY)
print(f"Provenance dictionary: {len(provenance_df)} documented columns.")
provenance_df

Provenance dictionary: 33 documented columns.


,column,source,type,description,transformation
0,ID_EVENTO_ATENCAO_SAUDE,DET,key,Unique identifier of the outpatient event.,"None -- it's the key used to merge DET, CONS, ..."
1,UF_PRESTADOR,DET,categorical,UF of the provider who performed the procedure.,Also exists in CONS; we validated that the val...
2,DT_REALIZACAO,DET,categorical/temporal,"Year-month of event occurrence, yyyy-mm format.","Same case as UF_PRESTADOR: identical to CONS, ..."
3,CD_PROCEDIMENTO,DET,categorical,Procedure/assistance item code (TUSS tables 19...,None. It will be the TUSS/SBOC filter key in t...
4,CD_TABELA_REFERENCIA,DET,categorical,TUSS reference table code for the procedure.,None.
5,QT_ITEM_EVENTO_INFORMADO,DET,numerical,Quantity of the item/procedure reported by the...,"None at this stage -- remains as string, with ..."
6,VL_ITEM_EVENTO_INFORMADO,DET,numerical,Reported value of the item/procedure.,None at this stage -- remains as string with d...
7,VL_ITEM_PAGO_FORNECEDOR,DET,numerical,Value actually paid by the operator directly t...,None at this stage -- remains as string with d...
8,UNIDADE_MEDIDA,DET,categorical,"Unit of measurement for the item, according to...",None.
9,IND_PACOTE,DET,binary categorical,Indicates whether the item is part of a proced...,"Maintained as string, in the original ANS stan..."


## Phase 3 — Reading, REM aggregation, merging and incremental writing, month by month

For each of the 12 months:

1. Reads DET, CONS, and REM from the zip downloaded. It remains this way
   until the end of the notebook, the ONLY exception is
   `VL_TOTAL_REMUNERACAO` from REM itself, converted to float only to be
   summed in step 2 below.
2. **Aggregates REM by event** before merging: sums `VL_TOTAL_REMUNERACAO`
   and concatenates `CD_MODELO_REMUNERACAO` with `"|"`, because a minority
   of events have more than one line in REM. Without this step, the merge
   would duplicate DET rows for these events.
3. Merges DET (base) + CONS by `ID_EVENTO_ATENCAO_SAUDE` (`left join`).
   `UF_PRESTADOR` and `DT_REALIZACAO` exist in both tables; we validated
   that the values match 100% between DET and CONS and discarded the CONS
   copy, to avoid loading a duplicated column.
4. Merges the result with the aggregated REM, also by
   `ID_EVENTO_ATENCAO_SAUDE` (`left join`, not every event has
   remuneration processed yet in the reference month, so nulls are
   expected here).
5. Adds `MES` and `UF`.
6. **Immediately writes the month to the final parquet**, directly at
   `OUTPUT_PATH` in `data/interim/` (via
   `pyarrow.parquet.ParquetWriter`, which appends each month as a new row
   group to the SAME file) and frees the month's memory (`del` +
   `gc.collect()`) before moving to the next.

**Why incremental writing, and not "merge all 12 months into a DataFrame
and only then save"?** A single month's base (DET+CONS+REM already merged)
occupies ~530MB in RAM with everything as a string. Storing all 12 months
simultaneously in a list and only then concatenating (`pd.concat`) requires
having ~6.4GB of monthly bases alive IN MEMORY at the same time, plus
another ~6.4GB for the new array that `pd.concat` allocates on top (the old
list is only freed later) a peak near 12GB, which exceeds a typical local
or Colab runtime. By writing month by month and freeing memory, the peak
remains at ~1 month of data (a few hundred MB), not the entire 12 months.

Quality checks (null key, duplicates, nulls by column) are performed HERE,
month by month, instead of needing the entire concatenated base in memory
later; duplicates between different months are impossible
anyway, because the `MES` column already differentiates the rows.

In [4]:
monthly_summary = []
nulls_by_column = None
total_duplicates = 0
parquet_writer = None

for m in MONTHS:
    det = read_csv_from_zip(zip_files["DET"][m])
    cons = read_csv_from_zip(zip_files["CONS"][m])
    rem = read_csv_from_zip(zip_files["REM"][m])

    # --- aggregate REM by event (see Phase 2 / Phase 3, item 2) ---
    rem["VL_TOTAL_REMUNERACAO"] = rem["VL_TOTAL_REMUNERACAO"].str.replace(",", ".", regex=False).astype(float)
    rem_agg = (
        rem.groupby("ID_EVENTO_ATENCAO_SAUDE")
        .agg(
            VL_TOTAL_REMUNERACAO=("VL_TOTAL_REMUNERACAO", "sum"),
            CD_MODELO_REMUNERACAO=("CD_MODELO_REMUNERACAO", lambda x: "|".join(sorted(set(x)))),
            N_LINHAS_REM=("ID_EVENTO_ATENCAO_SAUDE", "count"),
        )
        .reset_index()
    )

    # --- DET + CONS, with checking for duplicate columns before discarding ---
    base = det.merge(cons, on="ID_EVENTO_ATENCAO_SAUDE", how="left", suffixes=("", "_CONS_DUP"))
    dup_cols = [c for c in base.columns if c.endswith("_CONS_DUP")]
    for c in dup_cols:
        original = c.replace("_CONS_DUP", "")
        diverging_count = (base[original] != base[c]).sum()
        if diverging_count > 0:
            print(f"  [ATTENTION] month {m}: {diverging_count} rows with {original} diverging between DET and CONS")
    base = base.drop(columns=dup_cols)

    # --- + aggregated REM ---
    base = base.merge(rem_agg, on="ID_EVENTO_ATENCAO_SAUDE", how="left")

    base["MES"] = f"{YEAR}{m}"
    base["UF"] = STATE

    # --- month's diagnostics (read-only, nothing is corrected -- see Phase 4) ---
    base_row_count = len(base)
    monthly_nulls = base.isna().sum()
    nulls_by_column = monthly_nulls if nulls_by_column is None else nulls_by_column.add(monthly_nulls, fill_value=0)
    monthly_duplicates = int(base.duplicated().sum())
    total_duplicates += monthly_duplicates

    monthly_summary.append({
        "month": m,
        "det_rows": len(det),
        "det_unique_events": det["ID_EVENTO_ATENCAO_SAUDE"].nunique(),
        "base_rows": base_row_count,
        "null_key": int(base["ID_EVENTO_ATENCAO_SAUDE"].isna().sum()),
        "no_rem_match": int(base["VL_TOTAL_REMUNERACAO"].isna().sum()),
        "events_with_multiple_rem_rows": int((rem_agg["N_LINHAS_REM"] > 1).sum()),
        "duplicate_rows": monthly_duplicates,
    })
    print(f"[{STATE}] month {m}: DET={len(det):>9} rows -> base={base_row_count:>9} rows "
          f"(no match in REM: {int(base['VL_TOTAL_REMUNERACAO'].isna().sum())}, "
          f"duplicates: {monthly_duplicates})")

    # --- writes the month directly to the shared-folder parquet and frees memory ---
    table = pa.Table.from_pandas(base, preserve_index=False)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
    parquet_writer.write_table(table)

    del det, cons, rem, rem_agg, base, table
    gc.collect()

parquet_writer.close()

summary_df = pd.DataFrame(monthly_summary)
total_rows = int(summary_df["base_rows"].sum())
print(f"\nConsolidated base {STATE}/2025 (unfiltered): {total_rows:,} rows, "
      f"written incrementally to {OUTPUT_PATH}")

[AM] month 01: DET=  1461173 rows -> base=  1461173 rows (no match in REM: 7588, duplicates: 56045)


[AM] month 02: DET=  1383512 rows -> base=  1383512 rows (no match in REM: 8672, duplicates: 59850)


[AM] month 03: DET=  1350951 rows -> base=  1350951 rows (no match in REM: 12211, duplicates: 54357)


[AM] month 04: DET=  1389829 rows -> base=  1389829 rows (no match in REM: 8929, duplicates: 51257)


[AM] month 05: DET=  1397592 rows -> base=  1397592 rows (no match in REM: 9640, duplicates: 53096)


...

[AM] month 09: DET=  1436256 rows -> base=  1436256 rows (no match in REM: 9134, duplicates: 50459)


[AM] month 10: DET=  1409246 rows -> base=  1409246 rows (no match in REM: 8251, duplicates: 44461)


[AM] month 11: DET=  1271628 rows -> base=  1271628 rows (no match in REM: 6718, duplicates: 56539)


[AM] month 12: DET=  1003957 rows -> base=  1003957 rows (no match in REM: 5038, duplicates: 43132)



Consolidated base AM/2025 (unfiltered): 16,536,319 rows, written incrementally to data\interim\ans_ambulatorial_by_state\AM\AM_ambulatorial_full_2025.parquet


## Phase 4 — Structural validation

Before wrapping up, we check without correcting anything, just to know
what we are handing off to the North region treatment stage:

- If the key `ID_EVENTO_ATENCAO_SAUDE` never comes null (would indicate
  corruption in the source file).
- That the total number of rows in the final base matches the total number
  of DET rows summed over the 12 months (every `left join` must preserve
  the cardinality of DET, which is our base, if it doesn't match, some
  merge duplicated rows and needs to be investigated before exporting, not
  after).
- The null rate per column, only as a diagnostic (columns with very high
  rates may indicate a poorly filled field by operators, not a pipeline
  error), no nulls are filled or removed here.
- How many rows are 100% duplicated (all columns identical). This is only
  reported, NOT removed, deciding whether it's a "duplicate resubmission"
  or "same procedure performed twice for real" is a treatment decision,
  which is left for the complete North region base stage.

In [5]:
print("Summary by month:")
display(summary_df)

assert summary_df["null_key"].sum() == 0, "There are events without ID_EVENTO_ATENCAO_SAUDE!"
assert total_rows == int(summary_df["det_rows"].sum()), (
    "Number of rows in the final base diverges from the sum of DET -- some merge duplicated rows."
)
print("\nOK: no null keys, and the number of rows matches the total DET.")

null_rate = (nulls_by_column / total_rows * 100).sort_values(ascending=False)
print("\nNull rate per column (%, diagnostic only -- nothing is filled/removed here):")
display(null_rate[null_rate > 0].round(2))

print(f"\n100% duplicated rows (all columns identical), summing the 12 months: "
      f"{total_duplicates:,} ({total_duplicates / total_rows * 100:.2f}%). "
      "Kept as they are -- the decision to remove or not is left for the North region treatment. "
      "(Checked month by month, never between different months -- impossible to have duplicates across "
      "months, because the MES column already differentiates the rows.)")

Summary by month:


,month,det_rows,det_unique_events,base_rows,null_key,no_rem_match,events_with_multiple_rem_rows,duplicate_rows
0,01,1461173,698654,1461173,0,7588,1976,56045
1,02,1383512,679637,1383512,0,8672,2242,59850
2,03,1350951,666830,1350951,0,12211,1609,54357
3,04,1389829,708397,1389829,0,8929,1950,51257
4,05,1397592,720910,1397592,0,9640,2783,53096
5,06,1318800,695798,1318800,0,8783,2100,45832
6,07,1520258,776010,1520258,0,9602,2664,56664
7,08,1593117,769585,1593117,0,9103,2935,61597
8,09,1436256,736039,1436256,0,9134,2705,50459
9,10,1409246,746687,1409246,0,8251,2431,44461



OK: no null keys, and the number of rows matches the total DET.

Null rate per column (%, diagnostic only -- nothing is filled/removed here):


UNIDADE_MEDIDA               99.41
SAUDE_OCUPACIONAL            89.61
CD_MOTIVO_SAIDA              60.51
CBO                          58.28
TIPO_CONSULTA                52.57
CD_CARATER_ATENDIMENTO       30.28
TIPO_ATENDIMENTO             26.12
REGIME_ATENDIMENTO           15.58
IND_ACIDENTE_DOENCA          15.22
QT_ITEM_EVENTO_INFORMADO     12.45
VL_ITEM_PAGO_FORNECEDOR      12.45
VL_ITEM_EVENTO_INFORMADO     12.45
CD_MODELO_REMUNERACAO         0.63
N_LINHAS_REM                  0.63
VL_TOTAL_REMUNERACAO          0.63
CD_MUNICIPIO_BENEFICIARIO     0.01
SEXO                          0.01
dtype: float64


100% duplicated rows (all columns identical), summing the 12 months: 633,289 (3.83%). Kept as they are -- the decision to remove or not is left for the North region treatment. (Checked month by month, never between different months -- impossible to have duplicates across months, because the MES column already differentiates the rows.)


## Phase 5 — Confirming the export

The final parquet was already written in Phase 3, incrementally, month by
month, **directly into `data/interim/`**. We save in **parquet**, not
Excel/CSV because the unfiltered base of a single state already exceeds 1 million
rows per month, which exceeds the 1,048,576 row limit per Excel sheet.
Parquet also compresses much better than CSV even with all columns still as
string (dtype `object`), the actual typing only happens in the complete North region base.

In [6]:
print(f"Saved to: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB)")

provenance_df.to_csv(PROVENANCE_PATH, index=False)
print(f"Provenance table saved to: {PROVENANCE_PATH}")

Saved to: data\interim\ans_ambulatorial_by_state\AM\AM_ambulatorial_full_2025.parquet (183.1 MB)


Provenance table saved to: data\interim\ans_ambulatorial_by_state\AM\AM_proveniencia_colunas.csv


## Phase 6 — Detailed provenance (what came from CONS and REM to DET)

This section exists to directly document
everything that was passed from CONS and REM into the main table (DET
level).

In [7]:
print(f"Columns in the final base: {len(provenance_df)}")
print(f"Rows in the final base:  {total_rows:,}")

print("\n--- Original columns from DET (base, item/procedure level) ---")
display(provenance_df[provenance_df["source"] == "DET"][["column", "type", "description"]])

print("\n--- Columns that came from CONS (event context) ---")
display(provenance_df[provenance_df["source"] == "CONS"][["column", "type", "description"]])

print("\n--- Columns that came from REM (remuneration, aggregated by event) ---")
display(provenance_df[provenance_df["source"].str.startswith("REM")][["column", "type", "description", "transformation"]])

print("\n--- Columns derived from the ingestion process itself (do not exist in raw files) ---")
display(provenance_df[provenance_df["source"].str.startswith("derived")][["column", "type", "description", "transformation"]])

print(f"\nSummary: {(provenance_df['source'] == 'DET').sum()} columns from DET, "
      f"{(provenance_df['source'] == 'CONS').sum()} columns from CONS, "
      f"{provenance_df['source'].str.startswith('REM').sum()} columns from REM, "
      f"{provenance_df['source'].str.startswith('derived').sum()} columns derived from the process.")

Columns in the final base: 33
Rows in the final base:  16,536,319

--- Original columns from DET (base, item/procedure level) ---


,column,type,description
0,ID_EVENTO_ATENCAO_SAUDE,key,Unique identifier of the outpatient event.
1,UF_PRESTADOR,categorical,UF of the provider who performed the procedure.
2,DT_REALIZACAO,categorical/temporal,"Year-month of event occurrence, yyyy-mm format."
3,CD_PROCEDIMENTO,categorical,Procedure/assistance item code (TUSS tables 19...
4,CD_TABELA_REFERENCIA,categorical,TUSS reference table code for the procedure.
5,QT_ITEM_EVENTO_INFORMADO,numerical,Quantity of the item/procedure reported by the...
6,VL_ITEM_EVENTO_INFORMADO,numerical,Reported value of the item/procedure.
7,VL_ITEM_PAGO_FORNECEDOR,numerical,Value actually paid by the operator directly t...
8,UNIDADE_MEDIDA,categorical,"Unit of measurement for the item, according to..."
9,IND_PACOTE,binary categorical,Indicates whether the item is part of a proced...



--- Columns that came from CONS (event context) ---


,column,type,description
11,ID_PLANO,categorical,Plan identifier (valid only for this ANS publi...
12,FAIXA_ETARIA,categorical (ordinal),"Beneficiary's age group, in pre-defined bins b..."
13,SEXO,categorical,"Beneficiary's sex, according to TUSS table 43."
14,CD_MUNICIPIO_BENEFICIARIO,categorical,IBGE code of the beneficiary's municipality of...
15,PORTE,categorical (ordinal),Operator size by number of active beneficiarie...
16,CD_MODALIDADE,categorical,"Operator modality code (cooperative, self-mana..."
17,NM_MODALIDADE,categorical,Name of the operator modality.
18,CD_MUNICIPIO_PRESTADOR,categorical,IBGE code of the executive provider's municipa...
19,CD_CARATER_ATENDIMENTO,categorical,Nature of care: elective or urgency/emergency.
20,TIPO_ATENDIMENTO,categorical,"Type of care, according to TUSS table 50."



--- Columns that came from REM (remuneration, aggregated by event) ---


,column,type,description,transformation
28,VL_TOTAL_REMUNERACAO,numerical,Total remuneration of the event.,THE ONLY type conversion in this notebook (eve...
29,CD_MODELO_REMUNERACAO,categorical,Remuneration model(s) applied to the event.,When there is more than one model for the same...
30,N_LINHAS_REM,numerical,Number of REM rows aggregated for this event.,"New column, created at this stage. Flags event..."



--- Columns derived from the ingestion process itself (do not exist in raw files) ---


,column,type,description,transformation
31,MES,categorical,Competence (YYYYMM) of the source file.,"New column, assigned from the downloaded filen..."
32,UF,categorical,Acronym of the state processed in this notebook.,New column. It is the column used to concatena...



Summary: 11 columns from DET, 17 columns from CONS, 3 columns from REM, 2 columns derived from the process.


## How to replicate this notebook for another state

1. In Phase 0, replace `STATE = "AM"` with your state's acronym (e.g.,
   `"PA"`, `"RR"`).
2. Run all cells in sequence (Runtime > Run all).
3. Check Phase 4 (Structural validation), if any assert fails, stop and
   investigate before trusting the output; do not adjust the assert to
   "pass". The number of duplicates and null rate there are only
   diagnostic, they do not generate action in this notebook.
4. The file `data/interim/ans_ambulatorial_by_state/{YOUR_STATE}/{YOUR_STATE}_ambulatorial_full_2025.parquet`
   is ready for `02_merge_north_states.ipynb` with no manual file
   move needed.

## References

1. National Supplementary Health Agency (ANS). ANS Open Data - Open Data Portal (PDA): TISS, TUSS and beneficiaries [Internet]. Brasilia: ANS; 2026 [cited 2026 Aug 14]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/
2. National Supplementary Health Agency (ANS). Variable dictionary of the TISS Standard from 2023. Brasilia: ANS; 2024.